#Preprocess for all signals

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import mne

mne.set_log_level("WARNING")


BASE_DIR = Path(r"C:\Users\LENOVO\Desktop\sen\sen\SeizeIT2\SeizeIT2")  # BIDS root عندك

OUT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_ALL_MODALITIES")
OUT_EEG  = OUT_ROOT / "EEG"
OUT_ECG  = OUT_ROOT / "ECG"
OUT_EMG  = OUT_ROOT / "EMG"
OUT_MOV  = OUT_ROOT / "MOV"
for d in [OUT_EEG, OUT_ECG, OUT_EMG, OUT_MOV]:
    d.mkdir(parents=True, exist_ok=True)

WINDOW_SEC   = 4.0
PREICTAL_SEC = 15 * 60  # 15 minutes

EEG_NOTCH = 50
EEG_HP, EEG_LP = 0.5, 45
ECG_NOTCH = 50
ECG_HP, ECG_LP = 0.5, 40
EMG_NOTCH = 50
EMG_HP, EMG_LP = 40, 50
MOV_HP, MOV_LP = 0.3, 10.0


def parse_bids_ids_from_stem(stem: str):
    """
    Robust parsing for filenames like:
      sub-002_ses-01_task-szMonitoring_run-01_eeg
      sub-002_ses-01_task-szMonitoring_run-01_events
    """
    subject_m = re.search(r"(sub-\d+)", stem)
    session_m = re.search(r"(ses-\d+)", stem)
    run_m     = re.search(r"run-(\d+)", stem)

    if not subject_m or not session_m or not run_m:
        return None

    subject = subject_m.group(1)
    session = session_m.group(1)
    run     = int(run_m.group(1))
    return subject, session, run


def collect_all_runs_from_eeg_files(base_dir: Path):
    rows = []
    for eeg_path in base_dir.rglob("*_eeg.edf"):
        ids = parse_bids_ids_from_stem(eeg_path.stem)
        if ids is None:
            continue
        subject, session, run = ids
        rows.append({"subject": subject, "session": session, "run": run})

    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


def collect_seizure_events_from_bids(base_dir: Path, seizure_prefix=("sz",)):
    rows = []
    for ev_path in base_dir.rglob("*_events.tsv"):
        ids = parse_bids_ids_from_stem(ev_path.stem)
        if ids is None:
            continue
        subject, session, run = ids

        try:
            df = pd.read_csv(ev_path, sep="\t")
        except:
            continue

        if not {"onset", "duration", "eventType"}.issubset(df.columns):
            continue

        df["eventType"] = df["eventType"].astype(str).str.strip().str.lower()
        df["onset"] = pd.to_numeric(df["onset"], errors="coerce")
        df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
        df = df.dropna(subset=["onset", "duration"])

        for pref in seizure_prefix:
            sz = df[df["eventType"].str.startswith(pref)]
            for _, r in sz.iterrows():
                rows.append({
                    "subject": subject,
                    "session": session,
                    "run": run,
                    "onset": float(r["onset"]),
                    "duration": float(r["duration"]),
                    "eventType": r["eventType"]
                })

    return pd.DataFrame(rows)


def build_state_mask_from_events(raw, subject, session, run, events_df, preictal_sec):
    sf = float(raw.info["sfreq"])
    n_samples = raw.n_times
    dur_sec = n_samples / sf

    mask = np.zeros(n_samples, dtype=np.int8)

    if events_df is None or events_df.empty:
        return mask

    sub_df = events_df[
        (events_df.subject == subject) &
        (events_df.session == session) &
        (events_df.run == run)
    ]
    if sub_df.empty:
        return mask

    sub_df = sub_df.sort_values("onset")

    onset_vals = sub_df["onset"].astype(float).to_numpy()
    dur_vals   = sub_df["duration"].astype(float).to_numpy()

    use_samples = False
    if len(onset_vals) > 0:
        if np.nanmax(onset_vals) > (dur_sec * 1.2):
            use_samples = True

    for _, r in sub_df.iterrows():
        onset = float(r["onset"])
        dur   = float(r["duration"])

        if use_samples:
            ictal_start = int(round(onset))

            if dur <= (dur_sec * 1.2):
                ictal_end = int(round(onset + dur * sf))
            else:
                ictal_end = int(round(onset + dur)) 
        else:
            ictal_start = int(round(onset * sf))
            ictal_end   = int(round((onset + dur) * sf))

        if ictal_end <= 0 or ictal_start >= n_samples:
            continue

        ictal_start = max(0, ictal_start)
        ictal_end   = min(n_samples, ictal_end)

        pre_start = max(0, ictal_start - int(round(preictal_sec * sf)))

        if pre_start < ictal_start:
            mask[pre_start:ictal_start] = np.maximum(mask[pre_start:ictal_start], 1)
        if ictal_start < ictal_end:
            mask[ictal_start:ictal_end] = 2

    return mask


def prepare_ref_for_regression(eeg_raw, ref_raw):
    if ref_raw is None:
        return None

    sf = eeg_raw.info["sfreq"]
    ref = ref_raw.copy()

    if ref.info["sfreq"] != sf:
        ref.resample(sfreq=sf)

    ref_data = ref.get_data()
    eeg_len  = eeg_raw.n_times
    min_len  = min(ref_data.shape[1], eeg_len)

    ref_data = ref_data[:, :min_len]

    if np.var(ref_data) < 1e-10:
        return None

    return ref_data


def has_ecg_artefact(eeg_raw, ecg_raw, corr_thresh=0.3):
    ref = prepare_ref_for_regression(eeg_raw, ecg_raw)
    if ref is None:
        return False

    eeg = eeg_raw.get_data()[:, :ref.shape[1]]
    ref_ch = ref[0]

    return np.max([
        abs(np.corrcoef(eeg[ch], ref_ch)[0, 1])
        for ch in range(eeg.shape[0])
    ]) >= corr_thresh


def has_emg_artefact(eeg_raw, emg_raw, corr_thresh=0.35):
    ref = prepare_ref_for_regression(eeg_raw, emg_raw)
    if ref is None:
        return False

    sf = eeg_raw.info["sfreq"]
    eeg_bp = mne.filter.filter_data(eeg_raw.get_data(), sf, 20, 45, verbose=False)[:, :ref.shape[1]]
    ref_bp = mne.filter.filter_data(ref[0], sf, 20, 45, verbose=False)

    return np.max([
        abs(np.corrcoef(eeg_bp[ch], ref_bp)[0, 1])
        for ch in range(eeg_bp.shape[0])
    ]) >= corr_thresh


def regression_artifact_removal(eeg_raw, ref_raw):
    ref = prepare_ref_for_regression(eeg_raw, ref_raw)
    if ref is None:
        return eeg_raw.copy()

    Y = eeg_raw.get_data()[:, :ref.shape[1]]  # match ref length
    X = np.vstack([ref, np.ones(ref.shape[1])]).T  # (T, k+1)

    beta, *_ = np.linalg.lstsq(X, Y.T, rcond=None)
    artefact = (X @ beta).T

    cleaned = Y - artefact
    out = eeg_raw.copy()

    out_data = out.get_data()
    out_data[:, :ref.shape[1]] = cleaned
    out._data = out_data
    return out


def window_and_label_raw(raw, state_mask, win_sec):
    sf = raw.info["sfreq"]
    win = int(win_sec * sf)
    data = raw.get_data()
    n_samples = data.shape[1]

    X, y = [], []
    for s in range(0, n_samples - win + 1, win):
        e = s + win
        seg_mask = state_mask[s:e]

        if np.any(seg_mask == 2):
            lab = 2
        elif np.any(seg_mask == 1):
            lab = 1
        else:
            lab = 0

        X.append(data[:, s:e])
        y.append(lab)

    if len(X) == 0:
        return None, None

    return np.stack(X), np.array(y)


def window_and_label_array(data_chn, state_mask, sfreq, win_sec):
    win = int(win_sec * sfreq)
    n_samples = data_chn.shape[1]

    X, y = [], []
    for s in range(0, n_samples - win + 1, win):
        e = s + win
        seg_mask = state_mask[s:e]

        if np.any(seg_mask == 2):
            lab = 2
        elif np.any(seg_mask == 1):
            lab = 1
        else:
            lab = 0

        X.append(data_chn[:, s:e])
        y.append(lab)

    if len(X) == 0:
        return None, None

    return np.stack(X), np.array(y)


def safe_pick_channels(ch_names, include_keywords):
    picked = [i for i, n in enumerate(ch_names) if all(k in n for k in include_keywords)]
    return picked if len(picked) > 0 else None

def extract_or_nan(idx_list, data_arr, n_samples):
    if idx_list is None:
        return np.full((1, n_samples), np.nan, dtype=float)
    return data_arr[idx_list, :]

def full_mag(arr_kxn):
    return np.linalg.norm(arr_kxn, axis=0)

def make_4ch_magnitude(raw):
    data = raw.get_data()
    ch_names = raw.ch_names
    n_samples = raw.n_times

    EEG_ACC_idx = safe_pick_channels(ch_names, ["EEG SD", "ACC"])
    EEG_GYR_idx = safe_pick_channels(ch_names, ["EEG SD", "GYR"])
    ECG_ACC_idx = safe_pick_channels(ch_names, ["ECGEMG SD", "ACC"])
    ECG_GYR_idx = safe_pick_channels(ch_names, ["ECGEMG SD", "GYR"])

    EEG_ACC_data = extract_or_nan(EEG_ACC_idx, data, n_samples)
    EEG_GYR_data = extract_or_nan(EEG_GYR_idx, data, n_samples)
    ECG_ACC_data = extract_or_nan(ECG_ACC_idx, data, n_samples)
    ECG_GYR_data = extract_or_nan(ECG_GYR_idx, data, n_samples)

    mags = np.vstack([
        full_mag(EEG_ACC_data)[None, :],
        full_mag(EEG_GYR_data)[None, :],
        full_mag(ECG_ACC_data)[None, :],
        full_mag(ECG_GYR_data)[None, :],
    ])

    names = np.array(
        ["EEGSD_ACC_MAG", "EEGSD_GYR_MAG", "ECGEMG_ACC_MAG", "ECGEMG_GYR_MAG"],
        dtype=object
    )
    return mags, names


def process_and_save_eeg(subject, session, run, events_df):
    base = f"{subject}_{session}_task-szMonitoring_run-{run:02d}"
    eeg_path = BASE_DIR / subject / session / "eeg" / f"{base}_eeg.edf"
    ecg_path = BASE_DIR / subject / session / "ecg" / f"{base}_ecg.edf"
    emg_path = BASE_DIR / subject / session / "emg" / f"{base}_emg.edf"

    if not eeg_path.exists():
        return False

    out = OUT_EEG / f"{subject}_{session}_run-{run:02d}_win{int(WINDOW_SEC)}s_pre{int(PREICTAL_SEC)}s_EEG.npz"
    if out.exists():
        print(f"⏭️ Skip (exists): {out.name}")
        return True

    eeg = mne.io.read_raw_edf(eeg_path, preload=True, verbose=False)
    ecg = mne.io.read_raw_edf(ecg_path, preload=True, verbose=False) if ecg_path.exists() else None
    emg = mne.io.read_raw_edf(emg_path, preload=True, verbose=False) if emg_path.exists() else None

    eeg.notch_filter(EEG_NOTCH)
    eeg.filter(EEG_HP, EEG_LP)

    mask = build_state_mask_from_events(eeg, subject, session, run, events_df, PREICTAL_SEC)

    if ecg is not None and has_ecg_artefact(eeg, ecg):
        eeg = regression_artifact_removal(eeg, ecg)

    if emg is not None and has_emg_artefact(eeg, emg):
        eeg = regression_artifact_removal(eeg, emg)

    X, y = window_and_label_raw(eeg, mask, WINDOW_SEC)
    if X is None or y is None or y.size == 0:
        return False

    np.savez_compressed(
        out,
        X=X.astype(np.float32),
        y=y.astype(np.int8),
        sfreq=float(eeg.info["sfreq"]),
        ch_names=np.array(eeg.ch_names, dtype=object),
        subject=subject,
        session=session,
        run=run
    )
    print(f"✅ Saved {out.name} | labels {dict(zip(*np.unique(y, return_counts=True)))}")
    return True


def process_and_save_ecg(subject, session, run, events_df):
    base = f"{subject}_{session}_task-szMonitoring_run-{run:02d}"
    ecg_path = BASE_DIR / subject / session / "ecg" / f"{base}_ecg.edf"
    if not ecg_path.exists():
        return False

    out = OUT_ECG / f"{subject}_{session}_run-{run:02d}_win{int(WINDOW_SEC)}s_pre{int(PREICTAL_SEC)}s_ECG.npz"
    if out.exists():
        print(f"⏭️ Skip (exists): {out.name}")
        return True

    raw = mne.io.read_raw_edf(ecg_path, preload=True, verbose=False)
    raw.notch_filter(ECG_NOTCH)
    raw.filter(ECG_HP, ECG_LP)

    mask = build_state_mask_from_events(raw, subject, session, run, events_df, PREICTAL_SEC)

    X, y = window_and_label_raw(raw, mask, WINDOW_SEC)
    if X is None or y is None or y.size == 0:
        return False

    np.savez_compressed(
        out,
        X=X.astype(np.float32),
        y=y.astype(np.int8),
        sfreq=float(raw.info["sfreq"]),
        ch_names=np.array(raw.ch_names, dtype=object),
        subject=subject,
        session=session,
        run=run
    )
    print(f"✅ Saved {out.name} | labels {dict(zip(*np.unique(y, return_counts=True)))}")
    return True


def process_and_save_emg(subject, session, run, events_df):
    base = f"{subject}_{session}_task-szMonitoring_run-{run:02d}"
    emg_path = BASE_DIR / subject / session / "emg" / f"{base}_emg.edf"
    if not emg_path.exists():
        return False

    out = OUT_EMG / f"{subject}_{session}_run-{run:02d}_win{int(WINDOW_SEC)}s_pre{int(PREICTAL_SEC)}s_EMG.npz"
    if out.exists():
        print(f"⏭️ Skip (exists): {out.name}")
        return True

    raw = mne.io.read_raw_edf(emg_path, preload=True, verbose=False)
    raw.notch_filter(EMG_NOTCH)
    raw.filter(EMG_HP, EMG_LP)

    mask = build_state_mask_from_events(raw, subject, session, run, events_df, PREICTAL_SEC)

    X, y = window_and_label_raw(raw, mask, WINDOW_SEC)
    if X is None or y is None or y.size == 0:
        return False

    np.savez_compressed(
        out,
        X=X.astype(np.float32),
        y=y.astype(np.int8),
        sfreq=float(raw.info["sfreq"]),
        ch_names=np.array(raw.ch_names, dtype=object),
        subject=subject,
        session=session,
        run=run
    )
    print(f"✅ Saved {out.name} | labels {dict(zip(*np.unique(y, return_counts=True)))}")
    return True


def process_and_save_mov(subject, session, run, events_df):
    base = f"{subject}_{session}_task-szMonitoring_run-{run:02d}"
    mov_path = BASE_DIR / subject / session / "mov" / f"{base}_mov.edf"
    if not mov_path.exists():
        return False

    out = OUT_MOV / f"{subject}_{session}_run-{run:02d}_win{int(WINDOW_SEC)}s_pre{int(PREICTAL_SEC)}s_MOV.npz"
    if out.exists():
        print(f"⏭️ Skip (exists): {out.name}")
        return True

    raw = mne.io.read_raw_edf(mov_path, preload=True, verbose=False)

    # No notch for MOV
    raw.filter(MOV_HP, MOV_LP, verbose=False)

    mask = build_state_mask_from_events(raw, subject, session, run, events_df, PREICTAL_SEC)

    mags, mag_names = make_4ch_magnitude(raw)
    sf = float(raw.info["sfreq"])
    X, y = window_and_label_array(mags, mask, sf, WINDOW_SEC)
    if X is None or y is None or y.size == 0:
        return False

    np.savez_compressed(
        out,
        X=X.astype(np.float32),
        y=y.astype(np.int8),
        sfreq=sf,
        ch_names=mag_names,
        subject=subject,
        session=session,
        run=run
    )
    print(f"✅ Saved {out.name} | labels {dict(zip(*np.unique(y, return_counts=True)))}")
    return True

events_table = collect_seizure_events_from_bids(BASE_DIR)
runs = collect_all_runs_from_eeg_files(BASE_DIR)

ok_eeg = ok_ecg = ok_emg = ok_mov = 0

for _, r in runs.iterrows():
    subject, session, run = r.subject, r.session, int(r.run)

    if process_and_save_eeg(subject, session, run, events_table):
        ok_eeg += 1
    if process_and_save_ecg(subject, session, run, events_table):
        ok_ecg += 1
    if process_and_save_emg(subject, session, run, events_table):
        ok_emg += 1
    if process_and_save_mov(subject, session, run, events_table):
        ok_mov += 1

print("\n🎉 Done.")
print(f"EEG saved/skipped: {ok_eeg}/{len(runs)}")
print(f"ECG saved/skipped: {ok_ecg}/{len(runs)}")
print(f"EMG saved/skipped: {ok_emg}/{len(runs)}")
print(f"MOV saved/skipped: {ok_mov}/{len(runs)}")
print(f"📁 Output root: {OUT_ROOT}")

✅ Saved sub-001_ses-01_run-01_win4s_pre900s_EEG.npz | labels {np.int64(0): np.int64(16329)}
✅ Saved sub-001_ses-01_run-01_win4s_pre900s_ECG.npz | labels {np.int64(0): np.int64(16329)}
✅ Saved sub-001_ses-01_run-01_win4s_pre900s_EMG.npz | labels {np.int64(0): np.int64(16329)}
✅ Saved sub-001_ses-01_run-01_win4s_pre900s_MOV.npz | labels {np.int64(0): np.int64(16329)}
✅ Saved sub-001_ses-01_run-02_win4s_pre900s_EEG.npz | labels {np.int64(0): np.int64(4954)}
✅ Saved sub-001_ses-01_run-02_win4s_pre900s_ECG.npz | labels {np.int64(0): np.int64(4954)}
✅ Saved sub-001_ses-01_run-02_win4s_pre900s_EMG.npz | labels {np.int64(0): np.int64(4954)}
✅ Saved sub-001_ses-01_run-02_win4s_pre900s_MOV.npz | labels {np.int64(0): np.int64(4954)}
✅ Saved sub-001_ses-01_run-03_win4s_pre900s_EEG.npz | labels {np.int64(0): np.int64(16000), np.int64(1): np.int64(225), np.int64(2): np.int64(19)}
✅ Saved sub-001_ses-01_run-03_win4s_pre900s_ECG.npz | labels {np.int64(0): np.int64(16000), np.int64(1): np.int64(225), n

#Handling imbalance in train and validation only

#CAP normal runs to 1 hour BUT avoid impd periods (device adjustment)

In [ ]:


from pathlib import Path
import re, gc
import numpy as np
import pandas as pd
from collections import defaultdict


NPZ_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_ALL_MODALITIES")

MOD_DIRS = {
    "EEG": NPZ_ROOT / "EEG",
    "ECG": NPZ_ROOT / "ECG",
    "EMG": NPZ_ROOT / "EMG",
    "MOV": NPZ_ROOT / "MOV",
}
MODS = ["EEG", "ECG", "EMG", "MOV"]

BIDS_ROOT = Path(r"C:\Users\LENOVO\Desktop\sen\sen\SeizeIT2\SeizeIT2")

# Output split root
OUT_SPLIT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD")
OUT_TRAIN = OUT_SPLIT_ROOT / "TRAIN"
OUT_VAL   = OUT_SPLIT_ROOT / "VAL"
OUT_TEST  = OUT_SPLIT_ROOT / "TEST"
for split_dir in [OUT_TRAIN, OUT_VAL, OUT_TEST]:
    for m in MODS:
        (split_dir / m).mkdir(parents=True, exist_ok=True)


WINDOW_SEC = 4.0

CAP_NORMAL_SEC  = 60 * 60  # 1 hour
CAP_NORMAL_WINS = int(CAP_NORMAL_SEC / WINDOW_SEC)  # 900

CAP_SEED = 42  

# impd exclusion
IMPD_EVENT_TYPES = {"impd"}    
IMPD_BUFFER_SEC  = 60           
EXCLUDE_INITIAL_SEC = 10 * 60   
EXCLUDE_INITIAL_WINS = int(EXCLUDE_INITIAL_SEC / WINDOW_SEC)

CAP_NORMAL_IN_TEST = False


ID_RE = re.compile(r"(sub-\d+)_+(ses-\d+)_+run-(\d+)")

def parse_ids_from_name(fname: str):
    m = ID_RE.search(fname)
    if not m:
        return None
    return m.group(1), m.group(2), int(m.group(3))


def index_npz_files():
    idx = defaultdict(lambda: defaultdict(dict))
    for mod, d in MOD_DIRS.items():
        if not d.exists():
            continue
        for f in d.glob("*.npz"):
            ids = parse_ids_from_name(f.name)
            if ids is None:
                continue
            sub, ses, run = ids
            idx[sub][(ses, run)][mod] = f
    return idx


def load_npz_mmap(path: Path):
    npz = np.load(path, allow_pickle=True, mmap_mode="r")
    X = npz["X"]
    y = npz["y"]
    return X, y, npz

def get_sfreq_from_npz(npz_obj):
    try:
        if "sfreq" in npz_obj:
            return float(npz_obj["sfreq"])
    except Exception:
        pass
    return None

def build_templates(indexed):
    templates = {}
    for mod in MODS:
        found = False
        for sub in indexed:
            for (ses, run), files in indexed[sub].items():
                p = files.get(mod)
                if p is None:
                    continue
                try:
                    X, _, _ = load_npz_mmap(p)
                    if X.ndim == 3 and X.shape[0] > 0:
                        templates[mod] = (int(X.shape[1]), int(X.shape[2]))
                        found = True
                        break
                except Exception:
                    continue
            if found:
                break
    return templates


def load_impd_events(bids_root: Path) -> pd.DataFrame:
    rows = []
    for ev_path in bids_root.rglob("*_events.tsv"):
        stem = ev_path.stem  # e.g. sub-001_ses-01_task-..._run-01_events
        m_sub = re.search(r"(sub-\d+)", stem)
        m_ses = re.search(r"(ses-\d+)", stem)
        m_run = re.search(r"run-(\d+)", stem)
        if not (m_sub and m_ses and m_run):
            continue
        sub = m_sub.group(1)
        ses = m_ses.group(1)
        run = int(m_run.group(1))

        try:
            df = pd.read_csv(ev_path, sep="\t")
        except Exception:
            continue

        if not {"onset","duration","eventType"}.issubset(df.columns):
            continue

        df["eventType"] = df["eventType"].astype(str).str.strip().str.lower()
        df["onset"] = pd.to_numeric(df["onset"], errors="coerce")
        df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
        df = df.dropna(subset=["onset","duration"])

        df = df[df["eventType"].isin(IMPD_EVENT_TYPES)]
        if df.empty:
            continue

        rec_dur = None
        if "recordingDuration" in df.columns:
            try:
                rec_dur = float(df["recordingDuration"].iloc[0])
            except Exception:
                rec_dur = None

        for _, r in df.iterrows():
            rows.append({
                "subject": sub,
                "session": ses,
                "run": run,
                "onset": float(r["onset"]),
                "duration": float(r["duration"]),
                "eventType": str(r["eventType"]),
                "recordingDuration": rec_dur
            })

    return pd.DataFrame(rows)

def infer_events_units_and_make_forbidden_mask(ev_df_run: pd.DataFrame, n_wins: int, sfreq: float):
   
    forbidden = np.zeros(n_wins, dtype=bool)
    if ev_df_run is None or ev_df_run.empty:
        return forbidden

    dur_sec_est = n_wins * WINDOW_SEC
    dur_samp_est = dur_sec_est * sfreq if (sfreq is not None) else None

    rec_dur = None
    if "recordingDuration" in ev_df_run.columns:
        rec_dur = ev_df_run["recordingDuration"].dropna()
        rec_dur = float(rec_dur.iloc[0]) if len(rec_dur) > 0 else None

    onset_vals = ev_df_run["onset"].astype(float).to_numpy()
    max_onset = float(np.nanmax(onset_vals)) if len(onset_vals) > 0 else 0.0

    use_samples = False
    if (rec_dur is not None) and (sfreq is not None) and (dur_samp_est is not None):
        if abs(rec_dur - dur_samp_est) < abs(rec_dur - dur_sec_est):
            use_samples = True
    else:
        if max_onset > dur_sec_est * 1.2 and (sfreq is not None) and (max_onset <= (dur_sec_est * sfreq * 1.2)):
            use_samples = True

    for _, r in ev_df_run.iterrows():
        onset = float(r["onset"])
        dur   = float(r["duration"])

        if use_samples and (sfreq is not None):
            onset_sec = onset / sfreq
            if dur <= dur_sec_est * 1.2:
                dur_sec = dur  # seconds
            else:
                dur_sec = dur / sfreq  # samples
        else:
            onset_sec = onset
            dur_sec   = dur

        t0 = max(0.0, onset_sec - IMPD_BUFFER_SEC)
        t1 = onset_sec + dur_sec + IMPD_BUFFER_SEC

        w0 = int(np.floor(t0 / WINDOW_SEC))
        w1 = int(np.ceil(t1 / WINDOW_SEC))  
        w0 = max(0, w0)
        w1 = min(n_wins, w1)
        if w0 < w1:
            forbidden[w0:w1] = True

    return forbidden


def compute_keep_indices_from_eeg(files_dict, subject, session, run,
                                 impd_events_df,
                                 cap_normal_wins=CAP_NORMAL_WINS,
                                 cap_normal=True,
                                 rng=None):
    if "EEG" not in files_dict:
        return None, None

    X_eeg, y_eeg, npz_eeg = load_npz_mmap(files_dict["EEG"])
    if X_eeg.size == 0 or y_eeg.size == 0:
        return None, None

    uniq = set(np.unique(y_eeg).tolist())

    if uniq == {0}:
        n = int(y_eeg.shape[0])

        if not cap_normal:
            idx_keep = np.arange(n, dtype=int)
            return idx_keep, y_eeg

        if n <= cap_normal_wins:
            idx_keep = np.arange(n, dtype=int)
            return idx_keep, y_eeg

        if rng is None:
            rng = np.random.default_rng(CAP_SEED)

        sfreq = get_sfreq_from_npz(npz_eeg) or 256.0  # fallback
        ev_run = None
        if impd_events_df is not None and not impd_events_df.empty:
            ev_run = impd_events_df[
                (impd_events_df.subject == subject) &
                (impd_events_df.session == session) &
                (impd_events_df.run == run)
            ]
        forbidden = infer_events_units_and_make_forbidden_mask(ev_run, n_wins=n, sfreq=sfreq)

        forbidden[:min(EXCLUDE_INITIAL_WINS, n)] = True

        max_start = n - cap_normal_wins
        candidates = []
        for s in range(0, max_start + 1):
            if not forbidden[s:s+cap_normal_wins].any():
                candidates.append(s)

        if len(candidates) == 0:
           
            best_s = 0
            best_bad = cap_normal_wins + 1
            for s in range(0, max_start + 1):
                bad = int(forbidden[s:s+cap_normal_wins].sum())
                if bad < best_bad:
                    best_bad = bad
                    best_s = s
                    if best_bad == 0:
                        break
            start = best_s
        else:
            start = int(candidates[int(rng.integers(0, len(candidates)))])

        idx_keep = np.arange(start, start + cap_normal_wins, dtype=int)
        return idx_keep, y_eeg

    elif (1 in uniq) or (2 in uniq):
        idx_keep = np.where((y_eeg == 1) | (y_eeg == 2))[0]
        if idx_keep.size == 0:
            return None, None
        return idx_keep.astype(int), y_eeg

    return None, None


def select_mod_filtered(mod, files_dict, idx_keep, templates):
    n_keep = len(idx_keep)

    if mod not in files_dict:
        if mod not in templates:
            return None
        ch, samp = templates[mod]
        return np.zeros((n_keep, ch, samp), dtype=np.float32)

    X_mod, _, _ = load_npz_mmap(files_dict[mod])
    if X_mod.ndim != 3:
        return None

    n_win_mod = X_mod.shape[0]
    ch, samp  = int(X_mod.shape[1]), int(X_mod.shape[2])

    valid_mask = idx_keep < n_win_mod
    valid_idx = idx_keep[valid_mask]

    if valid_idx.size == 0:
        return np.zeros((n_keep, ch, samp), dtype=np.float32)

    X_sel = np.array(X_mod[valid_idx], dtype=np.float32, copy=True)

    nan_mask = np.isnan(X_sel)
    if nan_mask.any():
        X_sel[nan_mask] = 0.0

    missing = n_keep - X_sel.shape[0]
    if missing > 0:
        X_sel = np.concatenate([X_sel, np.zeros((missing, ch, samp), dtype=np.float32)], axis=0)

    return X_sel

# =========================
# 9) Subject split 70/10/20
# =========================
def split_subjects_70_10_20(subjects, seed=42):
    subjects = sorted(list(subjects))
    rng = np.random.default_rng(seed)
    rng.shuffle(subjects)

    n = len(subjects)
    if n == 0:
        return [], [], []
    if n == 1:
        return subjects, [], []
    if n == 2:
        return [subjects[0]], [], [subjects[1]]

    n_train = int(np.floor(0.70 * n))
    n_val   = int(np.floor(0.10 * n))

    n_val = max(1, n_val)
    n_test = n - n_train - n_val
    if n_test < 1:
        n_test = 1
        n_train = n - n_val - n_test

    return subjects[:n_train], subjects[n_train:n_train+n_val], subjects[n_train+n_val:]


def export_split_filtered(indexed, templates, subjects, out_dir: Path,
                          impd_events_df,
                          seed=CAP_SEED,
                          cap_normal=True):
    saved = 0
    dropped = 0
    rng = np.random.default_rng(seed)

    for sub in subjects:
        if sub not in indexed:
            continue

        for (ses, run), files in indexed[sub].items():
            idx_keep, y_eeg = compute_keep_indices_from_eeg(
                files_dict=files,
                subject=sub, session=ses, run=run,
                impd_events_df=impd_events_df,
                cap_normal_wins=CAP_NORMAL_WINS,
                cap_normal=cap_normal,
                rng=rng
            )
            if idx_keep is None:
                dropped += 1
                continue

            X_eeg, _, _ = load_npz_mmap(files["EEG"])
            X_eeg_sel = np.array(X_eeg[idx_keep], dtype=np.float32, copy=True)

            m = np.isnan(X_eeg_sel)
            if m.any():
                X_eeg_sel[m] = 0.0

            y_sel = np.array(y_eeg[idx_keep], dtype=np.int8, copy=True)

            X_out = {"EEG": X_eeg_sel}
            for mod in ["ECG", "EMG", "MOV"]:
                X_mod_sel = select_mod_filtered(mod, files, idx_keep, templates)
                if X_mod_sel is None:
                    if mod in templates:
                        ch, samp = templates[mod]
                        X_mod_sel = np.zeros((len(idx_keep), ch, samp), dtype=np.float32)
                    else:
                        X_mod_sel = np.zeros_like(X_eeg_sel)
                X_out[mod] = X_mod_sel

            base = f"{sub}_{ses}_run-{run:02d}_win4s_pre900s"
            for mod in MODS:
                out_path = out_dir / mod / f"{base}_{mod}.npz"
                np.savez_compressed(out_path, X=X_out[mod], y=y_sel)

            saved += 1
            del X_eeg_sel, y_sel, X_out
            gc.collect()

    print(f"✅ Exported runs: {saved}")
    print(f"🗑️ Dropped runs (no/empty EEG or no keep windows): {dropped}")


if __name__ == "__main__":
    indexed = index_npz_files()
    templates = build_templates(indexed)

    impd_events = load_impd_events(BIDS_ROOT)
    print(f"Loaded impd events rows: {len(impd_events)}")

    all_subjects = list(indexed.keys())
    train_subs, val_subs, test_subs = split_subjects_70_10_20(all_subjects, seed=42)

    OUT_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
    (OUT_SPLIT_ROOT / "train_subjects.txt").write_text("\n".join(train_subs), encoding="utf-8")
    (OUT_SPLIT_ROOT / "val_subjects.txt").write_text("\n".join(val_subs), encoding="utf-8")
    (OUT_SPLIT_ROOT / "test_subjects.txt").write_text("\n".join(test_subs), encoding="utf-8")

    print("Subjects:", len(all_subjects))
    print("Train:", len(train_subs), "Val:", len(val_subs), "Test:", len(test_subs))

    export_split_filtered(indexed, templates, train_subs, OUT_TRAIN, impd_events_df=impd_events, seed=CAP_SEED, cap_normal=True)
    export_split_filtered(indexed, templates, val_subs,   OUT_VAL,   impd_events_df=impd_events, seed=CAP_SEED, cap_normal=True)

    export_split_filtered(indexed, templates, test_subs,  OUT_TEST,  impd_events_df=impd_events, seed=CAP_SEED, cap_normal=CAP_NORMAL_IN_TEST)

Loaded impd events rows: 464
Subjects: 125
Train: 87 Val: 12 Test: 26
✅ Exported runs: 2316
🗑️ Dropped runs (no/empty EEG or no keep windows): 0
✅ Exported runs: 190
🗑️ Dropped runs (no/empty EEG or no keep windows): 0
✅ Exported runs: 344
🗑️ Dropped runs (no/empty EEG or no keep windows): 0


#RESAULT DISTRIBUTION

In [5]:
from pathlib import Path
import numpy as np

ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD")
SPLITS = ["TRAIN", "VAL", "TEST"]

def label_stats(split_name):
    eeg_dir = ROOT / split_name / "EEG"

    counts = {0: 0, 1: 0, 2: 0}
    total = 0

    for f in eeg_dir.glob("*.npz"):
        y = np.load(f, allow_pickle=True)["y"]
        unique, freq = np.unique(y, return_counts=True)
        for u, c in zip(unique, freq):
            u = int(u)
            c = int(c)
            if u in counts:
                counts[u] += c
                total += c
            else:
                total += c

    print("\n" + "="*60)
    print(f"📊 LABEL DISTRIBUTION — {split_name}")
    print(f"Total windows: {total:,}\n")

    for lbl in [0, 1, 2]:
        pct = (counts[lbl] / total * 100) if total > 0 else 0
        name = "Normal (0)" if lbl == 0 else f"Seizure ({lbl})"
        print(f"{name:15s}: {counts[lbl]:>10,}  |  {pct:6.2f}%")

for split in SPLITS:
    label_stats(split)


📊 LABEL DISTRIBUTION — TRAIN
Total windows: 1,859,504

Normal (0)     :  1,718,712  |   92.43%
Seizure (1)    :    129,702  |    6.98%
Seizure (2)    :     11,090  |    0.60%

📊 LABEL DISTRIBUTION — VAL
Total windows: 155,543

Normal (0)     :    140,514  |   90.34%
Seizure (1)    :     14,188  |    9.12%
Seizure (2)    :        841  |    0.54%

📊 LABEL DISTRIBUTION — TEST
Total windows: 1,412,689

Normal (0)     :  1,389,805  |   98.38%
Seizure (1)    :     21,282  |    1.51%
Seizure (2)    :      1,602  |    0.11%


In [8]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.


#CHEAK Cleanup done

In [5]:
from pathlib import Path

# =========================================================
# 1) ROOT PATH
# =========================================================
SPLIT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD")

CKPT_DIR = SPLIT_ROOT / "CNN_CKPTS"
PACKED_ROOT = SPLIT_ROOT / "PACKED_RUNS"
FEAT_ROOT = SPLIT_ROOT / "CNN_FEATURES_FOR_BILSTM"

# =========================================================
# 2) WHAT TO DELETE
# =========================================================
DELETE_OLD_CHECKPOINTS = True
DELETE_OLD_PACKED_RUNS = True
DELETE_OLD_FEATURES = True

# =========================================================
# 3) HELPERS
# =========================================================
def delete_npz_files(folder: Path):
    deleted = 0
    if folder.exists():
        for p in folder.glob("*.npz"):
            try:
                p.unlink()
                deleted += 1
            except Exception as e:
                print(f"⚠️ Could not delete {p}: {e}")
    return deleted

def delete_pt_files(folder: Path):
    deleted = 0
    if folder.exists():
        for p in folder.glob("*.pt"):
            try:
                p.unlink()
                deleted += 1
            except Exception as e:
                print(f"⚠️ Could not delete {p}: {e}")
    return deleted

# =========================================================
# 4) RUN
# =========================================================
total_deleted = 0

if DELETE_OLD_CHECKPOINTS:
    n = delete_pt_files(CKPT_DIR)
    total_deleted += n
    print(f"🗑️ Deleted checkpoints: {n}")

if DELETE_OLD_PACKED_RUNS:
    n_train = delete_npz_files(PACKED_ROOT / "TRAIN")
    n_val   = delete_npz_files(PACKED_ROOT / "VAL")
    n_test  = delete_npz_files(PACKED_ROOT / "TEST")
    n = n_train + n_val + n_test
    total_deleted += n
    print(f"🗑️ Deleted packed TRAIN: {n_train}")
    print(f"🗑️ Deleted packed VAL  : {n_val}")
    print(f"🗑️ Deleted packed TEST : {n_test}")

if DELETE_OLD_FEATURES:
    n_train = delete_npz_files(FEAT_ROOT / "TRAIN")
    n_val   = delete_npz_files(FEAT_ROOT / "VAL")
    n_test  = delete_npz_files(FEAT_ROOT / "TEST")
    n = n_train + n_val + n_test
    total_deleted += n
    print(f"🗑️ Deleted features TRAIN: {n_train}")
    print(f"🗑️ Deleted features VAL  : {n_val}")
    print(f"🗑️ Deleted features TEST : {n_test}")

print("\n✅ Cleanup done.")
print(f"Total deleted files: {total_deleted}")
print(f"Root: {SPLIT_ROOT}")

🗑️ Deleted checkpoints: 0
🗑️ Deleted packed TRAIN: 0
🗑️ Deleted packed VAL  : 0
🗑️ Deleted packed TEST : 0
🗑️ Deleted features TRAIN: 0
🗑️ Deleted features VAL  : 0
🗑️ Deleted features TEST : 0

✅ Cleanup done.
Total deleted files: 0
Root: C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD
